# Speech Denoising — Notebook 02: Preprocessing

Dataset: VoiceBank+DEMAND  
Goal: Load raw audio files, slice into 1-second chunks, save as NumPy arrays.

In [ ]:
import numpy as np
import librosa as lb
from pathlib import Path
from tqdm import tqdm

## Dataset Paths

In [ ]:
clean_trainset = Path("data/archive/clean_trainset_28spk_wav/")
noisy_trainset = Path("data/archive/noisy_trainset_28spk_wav/")
clean_testset  = Path("data/archive/clean_testset_wav/")
noisy_testset  = Path("data/archive/noisy_testset_wav/")

## File Lists

In [ ]:
clean_trainset_files = sorted(clean_trainset.glob("*.wav"))
noisy_trainset_files = sorted(noisy_trainset.glob("*.wav"))
clean_testset_files  = sorted(clean_testset.glob("*.wav"))
noisy_testset_files  = sorted(noisy_testset.glob("*.wav"))

## Chunking Function

Slices an audio array into fixed-size chunks. Last chunk is zero-padded if shorter than chunk_size.

In [ ]:
def slice_array(audio_array: np.ndarray, chunk_size: int) -> list:
    chunks = []
    for i in range(0, len(audio_array), chunk_size):
        chunk = audio_array[i: i + chunk_size]
        if len(chunk) < chunk_size:
            padding = np.zeros(chunk_size - len(chunk))
            chunk = np.concatenate([chunk, padding])
        chunks.append(chunk)
    return chunks

## Process All Splits

chunk_size = 16000 samples = 1 second at 16000 Hz

In [ ]:
CHUNK_SIZE = 16000

clean_testset_list = []
for file in tqdm(clean_testset_files, desc="clean testset"):
    audio, _ = lb.load(file, sr=None)
    clean_testset_list.append(slice_array(audio, CHUNK_SIZE))

noisy_testset_list = []
for file in tqdm(noisy_testset_files, desc="noisy testset"):
    audio, _ = lb.load(file, sr=None)
    noisy_testset_list.append(slice_array(audio, CHUNK_SIZE))

clean_trainset_list = []
for file in tqdm(clean_trainset_files, desc="clean trainset"):
    audio, _ = lb.load(file, sr=None)
    clean_trainset_list.append(slice_array(audio, CHUNK_SIZE))

noisy_trainset_list = []
for file in tqdm(noisy_trainset_files, desc="noisy trainset"):
    audio, _ = lb.load(file, sr=None)
    noisy_trainset_list.append(slice_array(audio, CHUNK_SIZE))

## Flatten to NumPy Arrays

In [ ]:
clean_testset_chunks = []
for file_chunks in clean_testset_list:
    clean_testset_chunks.extend(file_chunks)
clean_testset_chunks = np.array(clean_testset_chunks)

noisy_testset_chunks = []
for file_chunks in noisy_testset_list:
    noisy_testset_chunks.extend(file_chunks)
noisy_testset_chunks = np.array(noisy_testset_chunks)

clean_trainset_chunks = []
for file_chunks in clean_trainset_list:
    clean_trainset_chunks.extend(file_chunks)
clean_trainset_chunks = np.array(clean_trainset_chunks)

noisy_trainset_chunks = []
for file_chunks in noisy_trainset_list:
    noisy_trainset_chunks.extend(file_chunks)
noisy_trainset_chunks = np.array(noisy_trainset_chunks)

print(f"clean_testset_chunks:  {clean_testset_chunks.shape}")
print(f"noisy_testset_chunks:  {noisy_testset_chunks.shape}")
print(f"clean_trainset_chunks: {clean_trainset_chunks.shape}")
print(f"noisy_trainset_chunks: {noisy_trainset_chunks.shape}")

## Save to Disk

In [ ]:
np.save("data/processed/clean_testset_chunks.npy",  clean_testset_chunks)
np.save("data/processed/noisy_testset_chunks.npy",  noisy_testset_chunks)
np.save("data/processed/clean_trainset_chunks.npy", clean_trainset_chunks)
np.save("data/processed/noisy_trainset_chunks.npy", noisy_trainset_chunks)

print("Saved to data/processed/")